# Accessing Data

The data layer lives in the `src/data` module, accessed through `src.data.access`. This is a series of quick examples to get you up and running.

In [ ]:
# add the project root to the import path so `src.data` is importable
import sys
sys.path.insert(0, "../")  # replace with path/to/project/root
from src.data.access import get_data, get_site_ids, aggregate_by_interval

Site uids are the unique identifiers for water sites and their basins. We access the rest of the data by specifying one.

Let's see all of them:

In [ ]:
ids = get_site_ids()
print(f"Num sites: {len(ids)}")
print(ids[:5])

All sites follow the pattern `WQS*` (Iowa state sites) or `USGS-*` (USGS NWIS sites). Let's choose a USGS site (I picked one that I like) and look at its data.

In [ ]:
uid = "USGS-06604440"
data = get_data(site_uid=uid)  # basin, crops, grid, surplus, water, weather, basin_area, sensor_location

Quick list of the contents:
| Value | Description |
|-----------|------------|
| `data.basin` | the basin of the site. Probably don't need to access this directly, used mostly to compute the other pieces of data. |
| `data.crops` | crop info for crops in the basin |
| `data.weather` | per-cell daily weather for the basin (IEM precip + gridMET variables), from the site's start to end date (padded ±60 days). Precipitation is the `precip_in_1d` column. |
| `data.surplus` | nitrogen data inside the basin spanning 2000 till 2017. |
| `data.water` | the water timeseries data for the site |
| `data.grid` | the per-basin spatial grid: cell geometry, coordinates (`lon`/`lat`), area, etc. |

In [ ]:
data.water.info()

In [ ]:
# plot the water timeseries
import plotly.express as px

# the index of the water dataframe is the DateTime of the row
# its used as an index because it speeds up various DateTime operations

fig = px.line(
    data.water, 
    x=data.water.index, 
    y="nitrate_con")
fig.show()

Plotting the whole timeseries might make your notebook crash for long-lived sites because `data.water` is unaggregated; you're seeing the entire timeseries. Better to aggregate and then plot. Two options:

In [ ]:
# Option 1: aggregate manually
uid = "USGS-05482500" # 18 year lifespan, lots of data
water = get_data(site_uid=uid).water
nitrate = water["nitrate_con"].resample("1W").agg("mean")
    # aggregation easy bc the index is a datetime, just call resample and agg

fig = px.line(nitrate, x=nitrate.index, y="nitrate_con")
fig.show()


In [ ]:
# Option 2: use the aggregation helper from src.data.access
uid = "USGS-05482500"
nitrate_con = aggregate_by_interval(site_uid=uid, value_col="nitrate_con", interval="1W", agg_func="mean")

fig = px.line(nitrate_con)
fig.show()

## Example: aggregate weather data

The weather frame has one row per (cell, day), with `precip_in_1d` (IEM) plus the gridMET variables. Cells are keyed by `node_id` / `global_node_id` (coordinates live in `data.grid`, joinable on `global_node_id`). Aggregate it over time **per cell** with the `set_index` → `groupby` → `resample` → `reset_index` pattern below (there is no per-cell built-in helper -- `aggregate_by_interval` is for a single water series):

In [ ]:
uid = "USGS-05482500"
weather = get_data(uid).weather
weather.info()

In [ ]:
# Aggregate per cell over time: set_index -> groupby(cell) -> resample -> reset_index.
agg_weather = (
    weather.set_index("date")
    .groupby(["node_id", "global_node_id"])["precip_in_1d"]
    .resample("1W").agg("sum")
    .reset_index()
    .rename(columns={"precip_in_1d": "precip_1w"})
)
agg_weather.info()

Aggregate water and weather. The old `data.agg()` one-shot method was removed in the refactor, so aggregate each directly: water with `aggregate_by_interval`, weather with the per-cell pattern above.

In [ ]:
uid = "USGS-05482500"
water = aggregate_by_interval(site_uid=uid, value_col="nitrate_con", interval="1W", agg_func="mean")

weather = get_data(uid).weather
agg_weather = (
    weather.set_index("date")
    .groupby(["node_id", "global_node_id"])["precip_in_1d"]
    .resample("1W").agg("sum")
    .reset_index()
    .rename(columns={"precip_in_1d": "precip_1w"})
)
print(water.head())
print(agg_weather[["date", "precip_1w"]].info())

Notice the weather is a way larger dataframe. This is because we're aggregating only across time, we still have a row for every cell (`node_id`/`global_node_id`) in the basin. This is good for training models -- more features -- but bad for plotting. Here's a function that produces a plotly figure with combined precipitation and water data:

In [ ]:
import plotly.graph_objects as go

# expects water and rain aggregated as above
def plot_rain_water(water, rain):
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=rain["date"],
            y=rain["precip_1w"],
            name="Precip (in)",
            yaxis="y2",
            marker_color="#3a94fa",
        )
    )
    fig.update_layout(
        yaxis2=dict(
            title="Precipitation (in)",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
    )
    fig.add_trace(
        go.Scatter(
            x=water.index,
            y=water.values,
            name="N (mg/L)",
            yaxis="y1",
            marker_color="#0f8e5e",
        )
    )
    fig.update_layout(
        yaxis={"title": "Nitrate (mg/L)"},
        xaxis={"title": None},
        legend={"orientation": "h", "y": -0.15},
        margin={"t": 20, "b": 40, "l": 50, "r": 50},
    )
    return fig

Try this graph in two scenarios: once on the per-cell weekly precip (`agg_weather`) as-is, and once after averaging the per-cell precipitation down to one value per date:

In [ ]:
fig = plot_rain_water(water, agg_weather)
fig.show()

In [ ]:
# collapse the per-cell weather to one row per date (mean precip across cells)
print("cells:", agg_weather["node_id"].nunique())
weather_df = agg_weather.groupby("date")["precip_1w"].mean().reset_index()

fig = plot_rain_water(water=water, rain=weather_df)
fig.show()

There is also `surplus` data, that's maybe easier to explore in the widget and by inspecting the `data.surplus` DataFrame.

In [ ]:
uid = "WQS0003"

data = get_data(uid)
print(data.crops.columns)
print(data.surplus.columns)
print(data.grid.columns)

In [ ]:
print(data.grid.dtypes)

In [ ]:
years = set(range(2000, 2026))
bad_uids = []
for uid in get_site_ids():
    crops = get_data(uid).crops
    if years != set(crops.year.unique()):
        bad_uids.append(uid)
        print(f" {uid} bad: {crops.year.unique()}")

crops.year.unique()